In [1]:
# ============================================================
# FASE 5 — PREPROCESSING & CLEANSING
# Cell 1: Setup + Fix DFT-02 (Vibration Negatif) + DFT-01 Info
# ============================================================

import sys
import numpy as np
import pandas as pd
from pathlib import Path

# ----------------------------------------------------------
# [1] SYS.PATH SETUP
# Notebook: notebooks/fase_5_preprocessing/
# ML_ROOT  = dua level ke atas
# ----------------------------------------------------------
NOTEBOOK_DIR = Path().resolve()
ML_ROOT      = NOTEBOOK_DIR.parent.parent
SRC_PATH     = ML_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

# ----------------------------------------------------------
# [2] IMPORT CONFIG
# ----------------------------------------------------------
from config import DATA_INTERIM_DIR, DATA_PROCESSED_DIR, GLOBAL_SEED

np.random.seed(GLOBAL_SEED)

# ----------------------------------------------------------
# [3] LOAD DATA
# ----------------------------------------------------------
INPUT_PATH = DATA_INTERIM_DIR / "df_sensor_featured.parquet"

df = pd.read_parquet(INPUT_PATH)

df = (
    df
    .sort_values(["machine_id", "timestamp"], ascending=True)
    .reset_index(drop=True)
)

# ----------------------------------------------------------
# [4] KONFIRMASI SETUP
# ----------------------------------------------------------
SEP = "=" * 65
sep = "-" * 65

print(SEP)
print("  SETUP KONFIRMASI — FASE 5 PREPROCESSING")
print(SEP)
print(f"\n  SRC_PATH           : {SRC_PATH}")
print(f"  GLOBAL_SEED        : {GLOBAL_SEED}")
print(f"  INPUT_PATH         : {INPUT_PATH.name}")
print(f"  DATA_INTERIM_DIR   : {DATA_INTERIM_DIR}")
print(f"  DATA_PROCESSED_DIR : {DATA_PROCESSED_DIR}")
print(f"\n  Shape df           : {df.shape}")
print(f"  Jumlah kolom       : {df.shape[1]}")
print(f"\n  timestamp min      : {df['timestamp'].min()}")
print(f"  timestamp max      : {df['timestamp'].max()}")
print(SEP)

# ════════════════════════════════════════════════════════════
# FIX DFT-02 — VIBRATION NEGATIF
# ════════════════════════════════════════════════════════════

print(f"\n{SEP}")
print("  FIX DFT-02 — VIBRATION NEGATIF")
print(SEP)

AFFECTED_COLS = [
    "vibration",
    "vibration_roll_mean_24h", "vibration_roll_mean_48h",
    "vibration_roll_max_24h",  "vibration_roll_max_48h",
    "vibration_lag_6h", "vibration_lag_12h", "vibration_lag_24h",
    "temp_per_vibration", "noise_per_vibration",
]

RATIO_COLS = ["temp_per_vibration", "noise_per_vibration"]
CLIP_COLS  = [c for c in AFFECTED_COLS if c not in RATIO_COLS]

# Defensive: hanya proses kolom yang ada di df
affected_ok = [c for c in AFFECTED_COLS if c in df.columns]
clip_ok     = [c for c in CLIP_COLS     if c in df.columns]
ratio_ok    = [c for c in RATIO_COLS    if c in df.columns]

# ----------------------------------------------------------
# [A] SEBELUM FIX
# ----------------------------------------------------------
print(f"\n{sep}")
print("  [A] NILAI <= 0 SEBELUM FIX")
print(sep)
print()

for col in affected_ok:
    n = int((df[col] <= 0).sum())
    flag = " ← akan difix" if n > 0 else ""
    print(f"    {col:<40} : {n:>8,}{flag}")

# ----------------------------------------------------------
# [B] FIX: CLIP kolom sensor asli, rolling, lag
# ----------------------------------------------------------
for col in clip_ok:
    df[col] = df[col].clip(lower=0)

# ----------------------------------------------------------
# [C] FIX: HITUNG ULANG kolom ratio dari nilai bersih
# ----------------------------------------------------------
df["temp_per_vibration"]  = df["temperature"] / (df["vibration"] + 1e-9)
df["noise_per_vibration"] = df["noise_level"] / (df["vibration"] + 1e-9)

for col in ratio_ok:
    n_inf = int(np.isinf(df[col]).sum())
    if n_inf > 0:
        df[col]    = df[col].replace([np.inf, -np.inf], np.nan)
        col_median = df[col].median()
        df[col]    = df[col].fillna(col_median)
        print(f"\n  [INFO] {col}: {n_inf:,} Inf diganti median ({col_median:.6f})")

# ----------------------------------------------------------
# [D] SETELAH FIX: verifikasi nilai <= 0 (harus semua 0)
# ----------------------------------------------------------
print(f"\n{sep}")
print("  [B] NILAI <= 0 SETELAH FIX (harus semua 0)")
print(sep)
print()

all_clean = True
for col in affected_ok:
    n    = int((df[col] <= 0).sum())
    flag = "[OK]  " if n == 0 else "[WARN]"
    print(f"    {flag}  {col:<40} : {n:,}")
    if n > 0:
        all_clean = False

print()
if all_clean:
    print("  [OK] DFT-02 CLOSED — semua nilai vibration & turunannya >= 0.")
else:
    print("  [WARN] DFT-02 belum sepenuhnya resolved.")

# ════════════════════════════════════════════════════════════
# INFO DFT-01 — PARTS_REPLACED
# ════════════════════════════════════════════════════════════

print(f"\n{SEP}")
print("  INFO DFT-01 — PARTS_REPLACED")
print(SEP)

if "parts_replaced" not in df.columns:
    print("\n  [OK] DFT-01 CLOSED.")
    print("  Kolom 'parts_replaced' TIDAK ada di df sensor (tidak di-merge).")
    print("  Tidak ada dampak ke pipeline modeling.")
else:
    n_nan = int(df["parts_replaced"].isna().sum())
    print(f"\n  [INFO] Kolom 'parts_replaced' ada di df ({n_nan:,} NaN).")
    print("  Penanganan: fill → 'Unknown' di cell selanjutnya.")

# ════════════════════════════════════════════════════════════
# VALIDASI AKHIR
# ════════════════════════════════════════════════════════════

print(f"\n{SEP}")
print("  VALIDASI AKHIR CELL 1")
print(SEP)

total_nan = int(df.isna().sum().sum())
num_cols  = df.select_dtypes(include=[np.number]).columns
total_inf = int(np.isinf(df[num_cols].values).sum())
shape_ok  = df.shape == (100_000, 75)

print(f"\n  Total NaN di seluruh df     : {total_nan:,}  "
      f"→  {'[OK]' if total_nan == 0 else '[WARN]'}")
print(f"  Total Inf di kolom numerik  : {total_inf:,}  "
      f"→  {'[OK]' if total_inf == 0 else '[WARN]'}")
print(f"  Shape df                    : {df.shape}  "
      f"→  {'[OK]' if shape_ok else '[WARN] expected (100000, 75)'}")

print(f"\n{SEP}")
print("  [OK] Setup & Defect Fix selesai. df siap untuk Cell 2.")
print(SEP)


  SETUP KONFIRMASI — FASE 5 PREPROCESSING

  SRC_PATH           : C:\PORTFOLIO\PROJECTS\PBL\Predictive Maintenance\projects\predictive-maintenance-monorepo\machine_learning\src
  GLOBAL_SEED        : 42
  INPUT_PATH         : df_sensor_featured.parquet
  DATA_INTERIM_DIR   : C:\PORTFOLIO\PROJECTS\PBL\Predictive Maintenance\projects\predictive-maintenance-monorepo\machine_learning\data\interim
  DATA_PROCESSED_DIR : C:\PORTFOLIO\PROJECTS\PBL\Predictive Maintenance\projects\predictive-maintenance-monorepo\machine_learning\data\processed

  Shape df           : (100000, 75)
  Jumlah kolom       : 75

  timestamp min      : 2025-07-01 00:00:00
  timestamp max      : 2026-01-25 07:00:00

  FIX DFT-02 — VIBRATION NEGATIF

-----------------------------------------------------------------
  [A] NILAI <= 0 SEBELUM FIX
-----------------------------------------------------------------

    vibration                                :        2 ← akan difix
    vibration_roll_mean_24h                

In [2]:
# ============================================================
# FASE 5 — Cell Re-Validasi DFT-02
# Cek nilai STRICTLY NEGATIF (< 0), bukan <= 0
# Nilai 0 adalah VALID (mesin idle/berhenti)
# ============================================================

SEP = "=" * 65
sep = "-" * 65

AFFECTED_COLS = [
    "vibration",
    "vibration_roll_mean_24h", "vibration_roll_mean_48h",
    "vibration_roll_max_24h",  "vibration_roll_max_48h",
    "vibration_lag_6h", "vibration_lag_12h", "vibration_lag_24h",
    "temp_per_vibration", "noise_per_vibration",
]

# Defensive: hanya kolom yang ada di df
affected_ok = [c for c in AFFECTED_COLS if c in df.columns]

print(SEP)
print("  RE-VALIDASI DFT-02 — STRICTLY NEGATIF (< 0)")
print(SEP)
print("  Catatan: nilai 0 adalah VALID (mesin idle/berhenti)")
print(f"  Kolom dicek : {len(affected_ok)}")
print()
print(f"  {'Kolom':<40}  {'Negatif (<0)':>12}  {'Zero (=0)':>10}  Status")
print(f"  {'-'*75}")

total_negative = 0

for col in affected_ok:
    n_neg  = int((df[col] < 0).sum())
    n_zero = int((df[col] == 0).sum())
    status = "✅ OK" if n_neg == 0 else "🔴 MASIH ADA NEGATIF"
    print(f"  {col:<40}  {n_neg:>12,}  {n_zero:>10,}  {status}")
    total_negative += n_neg

print(f"\n{sep}")
print("  KESIMPULAN")
print(sep)

if total_negative == 0:
    print("\n  ✅ DFT-02 FULLY RESOLVED")
    print("  Tidak ada satupun nilai negatif pada kolom vibration & turunannya.")
    print("  Nilai zero (=0) adalah VALID — merepresentasikan mesin idle/berhenti.")
    print("  Clip ke 0 bekerja dengan benar. DFT-02 dinyatakan CLOSED.")
else:
    print(f"\n  🔴 MASIH ADA {total_negative:,} NILAI NEGATIF — perlu investigasi lanjutan.")
    print("  Periksa kolom yang ditandai di atas.")

print(f"\n{sep}")
print("  CEK SHAPE FINAL")
print(sep)
shape_ok = df.shape == (100_000, 75)
flag     = "[OK]" if shape_ok else "[WARN]"
print(f"\n  Shape df : {df.shape}  →  {flag}")
if not shape_ok:
    print("  Expected : (100000, 75)")

print(f"\n{SEP}")
print("  [OK] Re-Validasi DFT-02 selesai.")
print(SEP)


  RE-VALIDASI DFT-02 — STRICTLY NEGATIF (< 0)
  Catatan: nilai 0 adalah VALID (mesin idle/berhenti)
  Kolom dicek : 10

  Kolom                                     Negatif (<0)   Zero (=0)  Status
  ---------------------------------------------------------------------------
  vibration                                            0           2  ✅ OK
  vibration_roll_mean_24h                              0           0  ✅ OK
  vibration_roll_mean_48h                              0           0  ✅ OK
  vibration_roll_max_24h                               0           0  ✅ OK
  vibration_roll_max_48h                               0           0  ✅ OK
  vibration_lag_6h                                     0           2  ✅ OK
  vibration_lag_12h                                    0           2  ✅ OK
  vibration_lag_24h                                    0           2  ✅ OK
  temp_per_vibration                                   0           0  ✅ OK
  noise_per_vibration                             

5.2 Normalisasi & Standarisasi Fitur.

In [3]:
# ============================================================
# FASE 5 — Cell 4 (REFACTORED): Normalisasi Anti-Scaler-Leakage
# Scaler di-fit HANYA pada training machines (M-01–M-14)
# ============================================================

import sys, joblib
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler, LabelEncoder

NOTEBOOK_DIR = Path().resolve()
ML_ROOT      = NOTEBOOK_DIR.parent.parent
SRC_PATH     = ML_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from config import DATA_PROCESSED_DIR, MODELS_ML_DIR

SEP = "=" * 65
sep = "-" * 65

TRAIN_MACHINES = [
    "M-01","M-02","M-03","M-04","M-05",
    "M-06","M-07","M-08","M-09","M-10",
    "M-11","M-12","M-13","M-14",
]
VAL_MACHINES  = ["M-15","M-16","M-17"]
TEST_MACHINES = ["M-18","M-19","M-20"]

# ════════════════════════════════════════════════════════════
# BAGIAN 1 — DEFINISI KOLOM
# ════════════════════════════════════════════════════════════

print(SEP)
print("  BAGIAN 1 — DEFINISI KOLOM PER KATEGORI")
print(SEP)

COLS_IDENTIFIER  = ["timestamp", "machine_id"]
COLS_LABEL       = ["health_label", "health_label_confirmed",
                    "health_label_encoded", "failure"]
COLS_CATEGORICAL = ["damage_category"]
COLS_ORDINAL     = ["severity_score"]

EXCLUDE      = set(COLS_IDENTIFIER + COLS_LABEL + COLS_CATEGORICAL)
COLS_NUMERIC = [c for c in df.columns if c not in EXCLUDE]

leaked = [c for c in COLS_NUMERIC
          if c in set(COLS_IDENTIFIER + COLS_LABEL + COLS_CATEGORICAL)]

print(f"\n  Jumlah COLS_NUMERIC : {len(COLS_NUMERIC)}")
print(f"  Kebocoran kolom     : {'Tidak ada ✅' if not leaked else leaked}")

# ════════════════════════════════════════════════════════════
# BAGIAN 2 — LABEL ENCODING: damage_category
# ════════════════════════════════════════════════════════════

print(f"\n{SEP}")
print("  BAGIAN 2 — LABEL ENCODING: damage_category")
print(SEP)

le = LabelEncoder()
le.fit(df["damage_category"])
df["damage_category_encoded"] = le.transform(df["damage_category"])

encoding_map = {label: int(code)
                for code, label in enumerate(le.classes_)}

print(f"\n  Kolom baru : damage_category_encoded")
print(f"\n  Mapping encoding:")
for label, code in encoding_map.items():
    n = int((df["damage_category"] == label).sum())
    print(f"    {label:<18} →  {code}  ({n:,} baris)")

n_nan_enc = int(df["damage_category_encoded"].isna().sum())
print(f"\n  NaN setelah encoding : {n_nan_enc}  "
      f"→  {'[OK]' if n_nan_enc == 0 else '[WARN]'}")

# ════════════════════════════════════════════════════════════
# BAGIAN 3 — STANDARDSCALER (ANTI SCALER LEAKAGE)
# ════════════════════════════════════════════════════════════

print(f"\n{SEP}")
print("  BAGIAN 3 — STANDARDSCALER (ANTI SCALER LEAKAGE)")
print(SEP)

# [1] Tambahkan damage_category_encoded ke COLS_NUMERIC
if "damage_category_encoded" not in COLS_NUMERIC:
    COLS_NUMERIC.append("damage_category_encoded")

print(f"\n  COLS_NUMERIC final : {len(COLS_NUMERIC)} kolom")

# [2] KRITIS — Fit scaler HANYA pada training machines
df_train_only = df[df["machine_id"].isin(TRAIN_MACHINES)]

print(f"\n{sep}")
print("  KONFIRMASI FIT SCALER")
print(sep)
print(f"\n  ✅ Scaler di-fit HANYA pada M-01–M-14")
print(f"  Baris untuk fit : {len(df_train_only):,}")
print(f"  Machine IDs     : {TRAIN_MACHINES}")
print(f"  Val/Test machines TIDAK termasuk dalam fit scaler")

# [3] Fit scaler pada train only
scaler = StandardScaler()
scaler.fit(df_train_only[COLS_NUMERIC])

# [4] Transform SELURUH df
df_scaled              = df.copy()
df_scaled[COLS_NUMERIC] = scaler.transform(df[COLS_NUMERIC])

print(f"\n{sep}")
print("  TRANSFORM SUMMARY")
print(sep)
print(f"\n  Scaler di-fit pada    : M-01–M-14 (14 mesin)")
print(f"  Scaler di-transform   : seluruh df (20 mesin)")

# [5] Verifikasi per split
SAMPLE_COLS = [c for c in ["temperature", "vibration", "noise_level"]
               if c in df_scaled.columns]

splits_verify = [
    ("Train (M-01–M-14)", TRAIN_MACHINES,
     "mean ≈ 0, std ≈ 1 ✅ EXPECTED"),
    ("Val   (M-15–M-17)", VAL_MACHINES,
     "mean/std ≠ 0/1  ✅ EXPECTED (tidak di-fit)"),
    ("Test  (M-18–M-20)", TEST_MACHINES,
     "mean/std ≠ 0/1  ✅ EXPECTED (tidak di-fit)"),
]

print(f"\n{sep}")
print("  VERIFIKASI HASIL SCALING PER SPLIT")
print(sep)

for split_name, machines, note in splits_verify:
    df_split = df_scaled[df_scaled["machine_id"].isin(machines)]
    print(f"\n  {split_name}  |  {note}")
    print(f"  {'Kolom':<30}  {'Mean':>10}  {'Std':>10}")
    print(f"  {'-'*55}")
    for col in SAMPLE_COLS:
        print(f"  {col:<30}  "
              f"{df_split[col].mean():>10.6f}  "
              f"{df_split[col].std():>10.6f}")

print(f"\n{sep}")
print("  CATATAN PENTING")
print(sep)
print("\n  Val/Test mean/std tidak persis 0/1 karena")
print("  scaler tidak di-fit pada mereka.")
print("  Ini BENAR dan EXPECTED — bukan bug.")
print("  Ini adalah bukti bahwa SCALER LEAKAGE telah dicegah.")

# [6] Simpan scaler
MODELS_ML_DIR.mkdir(parents=True, exist_ok=True)
SCALER_PATH = MODELS_ML_DIR / "scaler.pkl"
joblib.dump(scaler, SCALER_PATH)
scaler_size_kb = SCALER_PATH.stat().st_size / 1024

print(f"\n{sep}")
print("  SCALER TERSIMPAN")
print(sep)
print(f"\n  Path        : {SCALER_PATH}")
print(f"  Ukuran file : {scaler_size_kb:.2f} KB")
print(f"  Fit pada    : {len(df_train_only):,} baris (train machines saja)")

# ════════════════════════════════════════════════════════════
# VALIDASI AKHIR
# ════════════════════════════════════════════════════════════

print(f"\n{SEP}")
print("  VALIDASI AKHIR CELL 4")
print(SEP)

total_nan  = int(df_scaled.isna().sum().sum())
shape_ok   = df_scaled.shape == (100_000, 76)

print(f"\n  Total NaN di df_scaled  : {total_nan:,}  "
      f"→  {'[OK]' if total_nan == 0 else '[WARN]'}")
print(f"  Shape df_scaled         : {df_scaled.shape}  "
      f"→  {'[OK]' if shape_ok else '[WARN] expected (100000, 76)'}")
print(f"  scaler.pkl ukuran       : {scaler_size_kb:.2f} KB")

# ════════════════════════════════════════════════════════════
# BAGIAN 4 — EXPORT
# ════════════════════════════════════════════════════════════

print(f"\n{SEP}")
print("  BAGIAN 4 — EXPORT")
print(SEP)

DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

PARQUET_PATH = DATA_PROCESSED_DIR / "df_model_ready.parquet"
CSV_PATH     = DATA_PROCESSED_DIR / "df_model_ready_preview.csv"

df_scaled.to_parquet(PARQUET_PATH, index=False)
parquet_mb = PARQUET_PATH.stat().st_size / (1024 ** 2)

df_scaled.to_csv(CSV_PATH, index=False)
csv_mb = CSV_PATH.stat().st_size / (1024 ** 2)

print(f"\n  Parquet : {PARQUET_PATH}")
print(f"  Shape   : {df_scaled.shape}  |  {parquet_mb:.2f} MB")
print(f"\n  CSV     : {CSV_PATH}")
print(f"  Shape   : {df_scaled.shape}  |  {csv_mb:.2f} MB")

print(f"\n{sep}")
print("  SCALER LEAKAGE FIX")
print(sep)
print("\n  Scaler di-fit hanya pada training machines (M-01–M-14).")
print("  Val & Test machines di-transform saja,")
print("  statistik mereka tidak mempengaruhi scaler.")

print(f"\n{SEP}")
print("  [OK] Normalisasi Anti-Scaler-Leakage selesai.")
print(f"  [OK] df_scaled : {df_scaled.shape[0]:,} baris × {df_scaled.shape[1]} kolom")
print(SEP)


  BAGIAN 1 — DEFINISI KOLOM PER KATEGORI

  Jumlah COLS_NUMERIC : 68
  Kebocoran kolom     : Tidak ada ✅

  BAGIAN 2 — LABEL ENCODING: damage_category

  Kolom baru : damage_category_encoded

  Mapping encoding:
    Electrical         →  0  (30,488 baris)
    Lubrication        →  1  (12,736 baris)
    Mechanical         →  2  (28,952 baris)
    Routine            →  3  (9,832 baris)
    Thermal            →  4  (3,800 baris)
    Unknown            →  5  (14,192 baris)

  NaN setelah encoding : 0  →  [OK]

  BAGIAN 3 — STANDARDSCALER (ANTI SCALER LEAKAGE)

  COLS_NUMERIC final : 69 kolom

-----------------------------------------------------------------
  KONFIRMASI FIT SCALER
-----------------------------------------------------------------

  ✅ Scaler di-fit HANYA pada M-01–M-14
  Baris untuk fit : 70,000
  Machine IDs     : ['M-01', 'M-02', 'M-03', 'M-04', 'M-05', 'M-06', 'M-07', 'M-08', 'M-09', 'M-10', 'M-11', 'M-12', 'M-13', 'M-14']
  Val/Test machines TIDAK termasuk dalam fit sca

5.3 Export Final Fase 5

In [4]:
# ============================================================
# FASE 5 — Cell 4: Export Final & Quality Gate
# ============================================================

import numpy as np
import sys
from pathlib import Path

NOTEBOOK_DIR = Path().resolve()
ML_ROOT      = NOTEBOOK_DIR.parent.parent
SRC_PATH     = ML_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from config import DATA_PROCESSED_DIR, MODELS_ML_DIR

SEP = "=" * 65
sep = "-" * 65

# ════════════════════════════════════════════════════════════
# BAGIAN 1 — FEATURE INVENTORY FINAL
# ════════════════════════════════════════════════════════════

print(SEP)
print("  BAGIAN 1 — FEATURE INVENTORY FINAL (df_scaled)")
print(SEP)

INV_IDENTIFIER  = ["timestamp", "machine_id"]
INV_LABEL       = ["health_label", "health_label_confirmed",
                   "health_label_encoded", "failure"]
INV_CATEGORICAL = ["damage_category", "damage_category_encoded"]

inv_id_ok   = [c for c in INV_IDENTIFIER  if c in df_scaled.columns]
inv_lbl_ok  = [c for c in INV_LABEL       if c in df_scaled.columns]
inv_cat_ok  = [c for c in INV_CATEGORICAL if c in df_scaled.columns]

exclude_set = set(INV_IDENTIFIER + INV_LABEL + INV_CATEGORICAL)
inv_numeric = [c for c in df_scaled.columns if c not in exclude_set]

inventory = {
    "Identifier"  : inv_id_ok,
    "Label"       : inv_lbl_ok,
    "Categorical" : inv_cat_ok,
    "Numeric"     : inv_numeric,
}

print(f"\n  {'Grup':<14} {'Jumlah':>7}")
print(f"  {'-'*25}")
for grup, cols in inventory.items():
    print(f"  {grup:<14} {len(cols):>7}")
print(f"  {'-'*25}")
print(f"  {'TOTAL':<14} {sum(len(v) for v in inventory.values()):>7}")
print(f"\n  Shape df_scaled : {df_scaled.shape}")

# ════════════════════════════════════════════════════════════
# BAGIAN 2 — FINAL QUALITY GATE
# ════════════════════════════════════════════════════════════

print(f"\n{SEP}")
print("  BAGIAN 2 — FINAL QUALITY GATE")
print(SEP)
print()

checks  = {}
details = {}

# [1] NaN
total_nan = int(df_scaled.isna().sum().sum())
checks["[1] Total NaN = 0"]           = total_nan == 0
details["[1] Total NaN = 0"]          = f"NaN ditemukan : {total_nan:,}"

# [2] Inf
num_cols  = df_scaled.select_dtypes(include=[np.number]).columns
total_inf = int(np.isinf(df_scaled[num_cols].values).sum())
checks["[2] Total Inf = 0"]           = total_inf == 0
details["[2] Total Inf = 0"]          = f"Inf ditemukan : {total_inf:,}"

# [3] Shape
shape_ok = df_scaled.shape == (100_000, 76)
checks["[3] Shape (100k × 76)"]       = shape_ok
details["[3] Shape (100k × 76)"]      = f"Shape aktual  : {df_scaled.shape}"

# [4] scaler.pkl exists
scaler_path = MODELS_ML_DIR / "scaler.pkl"
scaler_ok   = scaler_path.exists()
checks["[4] scaler.pkl tersimpan"]    = scaler_ok
details["[4] scaler.pkl tersimpan"]   = (
    f"Path : {scaler_path}" if scaler_ok
    else f"File tidak ditemukan di {scaler_path}"
)

# [5] Kolom target tersedia & nilai unik benar
target_col    = "health_label_encoded"
target_exists = target_col in df_scaled.columns
if target_exists:
    unique_vals   = sorted(df_scaled[target_col].dropna().unique().tolist())
    target_vals_ok = set(unique_vals) == {0, 1, 2}
else:
    unique_vals    = []
    target_vals_ok = False

checks["[5] Target kolom valid"]      = target_exists and target_vals_ok
details["[5] Target kolom valid"]     = (
    f"Nilai unik : {unique_vals}  (expected [0, 1, 2])"
    if target_exists
    else f"Kolom '{target_col}' tidak ditemukan di df_scaled"
)

# --- Print hasil ---
all_pass = all(checks.values())
for name, passed in checks.items():
    icon   = "PASS" if passed else "FAIL"
    detail = details[name]
    print(f"  [{icon}]  {name}")
    print(f"          {detail}")
    print()

print(sep)
if all_pass:
    print("\n  ✅ QUALITY GATE CLEARED — df_scaled siap untuk export.")
else:
    failed = [n for n, p in checks.items() if not p]
    print(f"\n  🔴 QUALITY GATE FAILED — {len(failed)} check tidak lulus:")
    for f in failed:
        print(f"     • {f}")

# ════════════════════════════════════════════════════════════
# BAGIAN 3 — EXPORT KE data/processed/
# ════════════════════════════════════════════════════════════

print(f"\n{SEP}")
print("  BAGIAN 3 — EXPORT KE data/processed/")
print(SEP)

DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

PARQUET_PATH = DATA_PROCESSED_DIR / "df_model_ready.parquet"
CSV_PATH     = DATA_PROCESSED_DIR / "df_model_ready_preview.csv"

# Export Parquet
df_scaled.to_parquet(PARQUET_PATH, index=False)
parquet_mb = PARQUET_PATH.stat().st_size / (1024 ** 2)

# Export CSV
df_scaled.to_csv(CSV_PATH, index=False)
csv_mb = CSV_PATH.stat().st_size / (1024 ** 2)

print(f"\n{sep}")
print("  [FORMAT 1] Parquet — Pipeline Resmi Modeling")
print(sep)
print(f"\n  Path   : {PARQUET_PATH}")
print(f"  Shape  : {df_scaled.shape}")
print(f"  Ukuran : {parquet_mb:.2f} MB")

print(f"\n{sep}")
print("  [FORMAT 2] CSV — Preview Manual")
print(sep)
print(f"\n  Path   : {CSV_PATH}")
print(f"  Shape  : {df_scaled.shape}")
print(f"  Ukuran : {csv_mb:.2f} MB")

print(f"\n{sep}")
print("  CATATAN")
print(sep)
print("\n  df_model_ready.parquet adalah input resmi")
print("  untuk Fase 6 (Imbalance Handling), Fase 7 (Splitting),")
print("  dan Fase 8 (Modeling).")
print("\n  df_model_ready_preview.csv hanya untuk inspeksi manual.")

print(f"\n{SEP}")
print("  [OK] Preprocessing Fase 5 SELESAI.")
print(f"  [OK] Output : df_model_ready.parquet  "
      f"({df_scaled.shape[0]:,} baris × {df_scaled.shape[1]} kolom)")
print(SEP)


  BAGIAN 1 — FEATURE INVENTORY FINAL (df_scaled)

  Grup            Jumlah
  -------------------------
  Identifier           2
  Label                4
  Categorical          2
  Numeric             68
  -------------------------
  TOTAL               76

  Shape df_scaled : (100000, 76)

  BAGIAN 2 — FINAL QUALITY GATE

  [PASS]  [1] Total NaN = 0
          NaN ditemukan : 0

  [PASS]  [2] Total Inf = 0
          Inf ditemukan : 0

  [PASS]  [3] Shape (100k × 76)
          Shape aktual  : (100000, 76)

  [PASS]  [4] scaler.pkl tersimpan
          Path : C:\PORTFOLIO\PROJECTS\PBL\Predictive Maintenance\projects\predictive-maintenance-monorepo\machine_learning\models\ml_track\scaler.pkl

  [PASS]  [5] Target kolom valid
          Nilai unik : [0, 1, 2]  (expected [0, 1, 2])

-----------------------------------------------------------------

  ✅ QUALITY GATE CLEARED — df_scaled siap untuk export.

  BAGIAN 3 — EXPORT KE data/processed/

--------------------------------------------------